This notebook tests all four models: PLS, Ridge, Lasso, Elastic Net
=> on the raw preprocessed data, before implementing any feature selection
To compare performance before and after feature selection
Many components were tested on PLS to finally find a suitable range for components (6-11)
Also many alphas were tested on Ridge, Lasso and Elastic Net, and the best alpha was chosen

We applied standard scaling on the data and compared performance before and after scaling
=> results: PLS and Ridge work better with Standard scaling, while Lasso and ELastic Net's performance drop when it's used
=> So later after implementing feature selection, and evaluating with 4 models: Standard scaling was applied only when PLS or RIdge were used

In [1]:
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score

In [6]:
# ============================================================
# Load Data
# ============================================================
X = np.loadtxt("waves.csv", delimiter=",")
Y = np.loadtxt("enzymes.csv", delimiter=",")

if Y.ndim == 1:
    Y = Y.reshape(-1, 1)
n_targets = Y.shape[1]
n_features = X.shape[1]

random_state = 42

In [3]:
print("Global feature summary:")
print("Min of mins:", np.min(np.min(X, axis=0)))
print("Max of maxs:", np.max(np.max(X, axis=0)))
print("Mean of means:", np.mean(np.mean(X, axis=0)))
print("Mean of stds:", np.mean(np.std(X, axis=0)))

Global feature summary:
Min of mins: -31.389899
Max of maxs: 31.043804
Mean of means: 1.0582144154523315e-10
Mean of stds: 0.17245981446136863


Apply standard scaling

In [3]:
from sklearn.discriminant_analysis import StandardScaler

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# --- 2. Scale Y ---
scaler_Y = StandardScaler()
Y_scaled = scaler_Y.fit_transform(Y)


PLS performance with and without Standard Scaling (performance drops without)

In [13]:
pls = PLSRegression(n_components=8, scale=True)
pls.fit(X, Y)
Y_pred = pls.predict(X)
print(r2_score(Y, Y_pred, multioutput="raw_values"))


[0.59818028 0.59012708 0.42042678 0.62390697 0.17938592]


In [15]:
pls = PLSRegression(n_components=8, scale=False)
pls.fit(X, Y)
Y_pred = pls.predict(X)
print(r2_score(Y, Y_pred, multioutput="raw_values"))


[0.41721709 0.48381671 0.04988864 0.43307782 0.07002276]


Ridge model with and without Standard scaling (performance drops without)

In [7]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

#  --- 3. Fit Ridge on scaled data ---
ridge_model = Ridge(alpha=50.0, random_state=random_state)
ridge_model.fit(X_scaled, Y_scaled)

# --- 4. Predict Y (still scaled) ---
Y_pred_scaled = ridge_model.predict(X_scaled)

# --- 5. Inverse-transform predictions to original units ---
Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)

# --- 6. Compute R² in raw units ---
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[0.9278433  0.91401051 0.94045798 0.92131067 0.91606026]


In [19]:
ridge_model = Ridge(alpha=50.0, random_state=random_state)
ridge_model.fit(X, Y)

# --- 4. Predict Y (still scaled) ---
Y_pred = ridge_model.predict(X)

# --- 6. Compute R² in raw units ---
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[0.49223874 0.51009555 0.38130894 0.47644742 0.33736106]


Lasso model (performance drops WITH standard scaling)

In [8]:
#  --- 3. Fit Lasso on scaled data ---
lasso_model = Lasso(alpha=50.0, max_iter=30000, random_state=random_state)
lasso_model.fit(X_scaled, Y_scaled)

# --- 4. Predict Y (still scaled) ---
Y_pred_scaled = lasso_model.predict(X_scaled)

# --- 5. Inverse-transform predictions to original units ---
Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)

# --- 6. Compute R² in raw units ---
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[ 1.11022302e-16  2.22044605e-16  0.00000000e+00  0.00000000e+00
 -2.22044605e-16]


In [ ]:
lasso_model = Lasso(alpha=50.0, max_iter=30000, random_state=random_state)
lasso_model.fit(X, Y)
Y_pred = lasso_model.predict(X)
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[ 3.18396150e-01  4.05067720e-01  7.48229711e-02  3.32603252e-01
 -2.22044605e-16]


Elastic Net model (performance drops WITH standard scaling)

In [11]:
#  --- 3. Fit Elastic Net on scaled data ---
ElasticNet_model = ElasticNet(alpha=50.0, max_iter=30000, random_state=random_state)
ElasticNet_model.fit(X_scaled, Y_scaled)

# --- 4. Predict Y (still scaled) ---
Y_pred_scaled = ElasticNet_model.predict(X_scaled)
# --- 5. Inverse-transform predictions to original units ---
Y_pred = scaler_Y.inverse_transform(Y_pred_scaled)

# --- 6. Compute R² in raw units ---
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[ 1.11022302e-16  2.22044605e-16  0.00000000e+00  0.00000000e+00
 -2.22044605e-16]


In [12]:
elastic_model = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=30000,
                           random_state=random_state)
elastic_model.fit(X, Y)
Y_pred = elastic_model.predict(X)
print(r2_score(Y, Y_pred, multioutput="raw_values"))

[0.39843768 0.42332791 0.25021653 0.37318576 0.0012754 ]
